In [ ]:
from transformers import  RobertaPreTrainedModel
from transformers.models.roberta.modeling_roberta import RobertaModel, RobertaClassificationHead, RobertaConfig, RobertaPreTrainedModel
import torch.nn as nn
class RobertaForSequenceClassification(RobertaPreTrainedModel):
    def __init__(self, config: RobertaConfig):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config

        self.roberta = RobertaModel(config, add_pooling_layer=False)
        self.classifier = RobertaClassificationHead(config)

        # Initialize weights and apply final processing
        self.post_init()

    def forward(
        self,
        input_ids = None,
        attention_mask = None,
        token_type_ids = None,
        position_ids  = None,
        head_mask  = None,
        inputs_embeds= None,
        labels  = None,
        output_attentions = None,
        output_hidden_states  = None,
        return_dict = None,
    ) :
        r"""
        labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for computing the sequence classification/regression loss. Indices should be in `[0, ...,
            config.num_labels - 1]`. If `config.num_labels == 1` a regression loss is computed (Mean-Square loss), If
            `config.num_labels > 1` a classification loss is computed (Cross-Entropy).
        """
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.roberta(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        sequence_output = outputs[0]
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            # move labels to correct device to enable model parallelism
            labels = labels.to(logits.device)
            if self.config.problem_type is None:
                if self.num_labels == 1:
                    self.config.problem_type = "regression"
                elif self.num_labels > 1 and (labels.dtype == torch.long or labels.dtype == torch.int):
                    self.config.problem_type = "single_label_classification"
                else:
                    self.config.problem_type = "multi_label_classification"

            if self.config.problem_type == "regression":
                loss_fct = MSELoss()
                if self.num_labels == 1:
                    loss = loss_fct(logits.squeeze(), labels.squeeze())
                else:
                    loss = loss_fct(logits, labels)
            elif self.config.problem_type == "single_label_classification":
                loss_fct = CrossEntropyLoss()
                loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            elif self.config.problem_type == "multi_label_classification":
                loss_fct = BCEWithLogitsLoss()
                loss = loss_fct(logits, labels)

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )

In [ ]:
# model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)

In [1]:
from transformers import  RobertaPreTrainedModel
from transformers.models.roberta.modeling_roberta import RobertaModel, RobertaClassificationHead, RobertaConfig, RobertaPreTrainedModel
import torch.nn as nn

from torch.nn import BCEWithLogitsLoss, CrossEntropyLoss, MSELoss
from  transformers.modeling_outputs import  SequenceClassifierOutput
from transformers import AdamW, RobertaForSequenceClassification, RobertaConfig, RobertaModel
class MTLRobertaForSequenceClassification(RobertaForSequenceClassification):
    def __init__(self, config: RobertaConfig):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config
        # print(config)
        self.roberta = RobertaModel(config, add_pooling_layer=False)
        

        # if len(config.tasks) != config.mtl_task_num:
        #     raise Exception(f"Number of tasks is not equal to mtl task number") 
        self.classification_heads_dict = nn.ModuleDict()
        
        for task in  config.tasks:
            self.classification_heads_dict[task] = RobertaClassificationHead(config)
            
            self.reinit_classification_head(self.classification_heads_dict[task])
        
        # self.classifier = RobertaClassificationHead(config)

        # Initialize weights and apply final processing
        self.post_init()

    
    def forward(
        self,
        input_ids = None,
        attention_mask = None,
        token_type_ids = None,
        position_ids  = None,
        head_mask  = None,
        inputs_embeds= None,
        labels  = None,
        output_attentions = None,
        output_hidden_states  = None,
        return_dict = None,
    ) :
        r"""
        labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for computing the sequence classification/regression loss. Indices should be in `[0, ...,
            config.num_labels - 1]`. If `config.num_labels == 1` a regression loss is computed (Mean-Square loss), If
            `config.num_labels > 1` a classification loss is computed (Cross-Entropy).
        """
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict

        outputs = self.roberta(
            input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            position_ids=position_ids,
            head_mask=head_mask,
            inputs_embeds=inputs_embeds,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
        )
        sequence_output = outputs[0]
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            # move labels to correct device to enable model parallelism
            labels = labels.to(logits.device)
            if self.config.problem_type is None:
                if self.num_labels == 1:
                    self.config.problem_type = "regression"
                elif self.num_labels > 1 and (labels.dtype == torch.long or labels.dtype == torch.int):
                    self.config.problem_type = "single_label_classification"
                else:
                    self.config.problem_type = "multi_label_classification"

            if self.config.problem_type == "regression":
                loss_fct = MSELoss()
                if self.num_labels == 1:
                    loss = loss_fct(logits.squeeze(), labels.squeeze())
                else:
                    loss = loss_fct(logits, labels)
            elif self.config.problem_type == "single_label_classification":
                loss_fct = CrossEntropyLoss()
                loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
            elif self.config.problem_type == "multi_label_classification":
                loss_fct = BCEWithLogitsLoss()
                loss = loss_fct(logits, labels)

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions,
        )
        
    def reinit_classification_head(self, classification_head):
        self.roberta._init_weights(classification_head.dense)
        self.roberta._init_weights(classification_head.out_proj)

    
    def set_classification_head(self, task_name):
        self.classification_head = self.classification_heads_dict[task_name]

/home/golazizi/MTL2DIS/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
from torch.utils.data import Dataset
import torch


class CustomDataset(Dataset):
    def __init__(self, input_ids, attention_mask, labels):
        self.input_ids = input_ids
        self.attention_mask = attention_mask
        self.targets = labels

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        input_id = self.input_ids[idx]
        attention_mask = self.attention_mask[idx]
        label = self.targets[idx]

        return input_id, attention_mask, label


# def get_dataset(tokenizer, texts, labels,):
#     encoded_texts = tokenizer(
#         texts, padding=True, truncation=True, return_tensors="pt")
#     input_ids = encoded_texts["input_ids"]
#     attention_mask = encoded_texts['attention_mask']
#     labels = torch.tensor(labels)
#     dataset = CustomDataset(input_ids, attention_mask, labels)
#     return dataset

In [10]:
import random
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence
from collections import defaultdict
import numpy as np
from torch.utils.data import TensorDataset, DataLoader, WeightedRandomSampler, Dataset
import torch
import pandas as pd


def trim_batch(
    input_ids,
    pad_token_id,
    attention_mask=None,
    axis=0,
):
    """Remove columns that are populated exclusively by pad_token_id"""
    keep_column_mask = input_ids.ne(pad_token_id).any(dim=axis)
    if attention_mask is None:
        return input_ids[:, keep_column_mask] if axis == 0 else input_ids[keep_column_mask, :]
    else:
        return (input_ids[:, keep_column_mask], attention_mask[:, keep_column_mask])


class MTLDataloader(DataLoader):
    def __init__(self, dummy_dataset, *args, **kwargs):
        data_loaders = kwargs.pop('data_loaders')
        mtl_bal_sampling = kwargs.pop('mtl_bal_sampling')
        self.task_keys = kwargs.pop('task_keys')
        self.task_num = kwargs.pop('task_num')
        self.pad_token_id = kwargs.pop('pad_token_id')
        self.sqrt = kwargs.pop('sqrt')
        super().__init__(dummy_dataset, *args, **kwargs)
        self.data_loaders = data_loaders
        self.mtl_bal_sampling = mtl_bal_sampling
        self.loader_len = None

        # just for random sampling of examples from a task
        self.task_iterators = [iter(x) for x in self.data_loaders]

    def __iter__(self):

        def mtl_data_iterator():
            draws = []
            for i in range(self.task_num):
                draws.extend([i] * len(self.data_loaders[i]))
            iterators = [iter(_) for _ in self.data_loaders]
            random.shuffle(draws)
            self.loader_len = len(draws)
            for loader_id in draws:
                iterator = iterators[loader_id]
                yield next(iterator)

        def mtl_bal_data_iterator():
            draws = []
            max_dataloader_len = max([len(x) for x in self.data_loaders])
            for i in range(self.task_num):
                if self.sqrt:
                    # x : max_dataloader_len = sqrt(len(x)) : sqrt(len(max_dataloader_len))
                    batch_num = int(
                        max_dataloader_len * (len(self.data_loaders[i]) ** 0.5) // (max_dataloader_len ** 0.5))
                    draws.extend([i] * batch_num)
                else:
                    draws.extend([i] * max_dataloader_len)
            iterators = [iter(_) for _ in self.data_loaders]
            random.shuffle(draws)
            self.loader_len = len(draws)
            for loader_id in draws:
                task_name = self.task_keys[loader_id]
                iterator = iterators[loader_id]
                try:
                    batch = next(iterator)
                except StopIteration:
                    iterators[loader_id] = iter(self.data_loaders[loader_id])
                    iterator = iterators[loader_id]
                    batch = next(iterator)
                yield (loader_id, task_name), batch

        if self.mtl_bal_sampling:
            return mtl_bal_data_iterator()
        else:
            return mtl_data_iterator()

    def get_full_batch_from_task(self, task_id, config):
        tmp_dataloader = DataLoader(self.data_loaders[task_id].dataset, batch_size=len(
            self.data_loaders[task_id].dataset))
        batch = next(iter(tmp_dataloader))
        cq_inputs_sample, cq_attention_mask_sample, ans_inputs_sample, ans_attention_mask_sample = [
            torch.stack(x, 0).transpose(0, 1).cuda() for x in batch[0:4]]
        cq_inputs_sample, cq_attention_mask_sample = trim_batch(cq_inputs_sample, config.pad_token_id,
                                                                cq_attention_mask_sample)
        ans_inputs_sample, ans_attention_mask_sample = trim_batch(ans_inputs_sample, config.pad_token_id,
                                                                  ans_attention_mask_sample)
        labels = batch[4].cuda()
        return cq_inputs_sample, cq_attention_mask_sample, ans_inputs_sample, ans_attention_mask_sample, labels

    def __len__(self):
        return self.loader_len
        # if self.mtl_bal_sampling:
        #    max_dataloader_len = max([len(x) for x in self.data_loaders])
        #    return max_dataloader_len * len(self.data_loaders)
        # else:
        #    return sum([len(_) for _ in self.data_loaders])


def get_dataset(args, task_key, split, tokenizer):
    data_file = f"./data/{args.dataset}/{args.label}/annotators/{task_key}/{split}.csv"
    df = pd.read_csv(data_file)
    #TODO: sampling
    # if split == 'train':
    #     df = df.sample(frac=0.5, random_state=0).reset_index(drop=True)
    #     print(len(df))
    texts = df[args.text_col].tolist()
    labels = df[args.label].tolist()

    encoded_texts = tokenizer(
        texts, padding=True, truncation=True, return_tensors="pt")
    input_ids = encoded_texts["input_ids"]
    attention_mask = encoded_texts['attention_mask']
    # labels = torch.tensor(labels)
    dataset = CustomDataset(input_ids, attention_mask, labels)

    return dataset





In [2]:
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

def mtl_train(args, model, mtl_train_dataloader, device, optimizer):
    model.to(device)
    model.train()
    for batch in mtl_train_dataloader:
        (model_task_id, model_task_name), batch = batch
        input_ids, attention_mask, labels = batch
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)
        optimizer.zero_grad()
        model.set_classification_head(model_task_name)
        outputs = model.forward(input_ids=input_ids,
                                attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
    return model, loss



def evaluate(model, test_loader, device, result_path=None, result_file_name='test_result.json'):
    # get predictions
    test_predictions, test_labels, test_loss = predict(
        test_loader, model, device)

    # Generate and save the evaluation result
    precision, recall, f1, _ = precision_recall_fscore_support(test_labels, test_predictions,
                                                               average='binary')
    auc = roc_auc_score(test_labels, test_predictions)

    # # import IPython; IPython.embed()
    # # exit()
    # if result_path:
    #     report_dict = {'precision': precision,
    #                    'recall': recall,
    #                    'f1': f1,
    #                    'auc': auc}
    #     with open(f"{result_path}/{result_file_name}", "w") as report_file:
    #         json.dump(report_dict, report_file, indent=4)
    return [precision, recall, f1, auc, test_loss]

def predict(data_loader, model, device):
    model.eval()
    predictions = []
    test_labels = []
    test_loss = 0
    with torch.no_grad():
        for batch in data_loader:
            input_ids, attention_mask, labels = batch
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)
            # TODO set classification head
            outputs = model.forward(
                input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            test_loss += loss.item()

            logits = outputs.logits
            predicted = torch.argmax(logits, dim=1)
            predictions.extend(predicted.cpu().numpy())
            test_labels.extend(labels.cpu().numpy())

    avg_loss = test_loss / len(data_loader)
    return predictions, test_labels, avg_loss

In [11]:
from torch.utils.data import DataLoader
from collections import defaultdict
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from mtl_dataloader import get_dataset, MTLDataloader

class Tasks:

    def __init__(self, args, task_keys, tokenizer, few_shot=False):
        self.args = args
        self.dataset = args.dataset
        self.k_shot = args.k_shot
        self.task_keys = task_keys
        self.tokenizer = tokenizer
        self.encoded_dataset = defaultdict(dict)
        self.data_loader_maps = defaultdict(dict)

        self.few_shot = few_shot

        # [ALIREZA] this is for balancing the dataset
        self.balance_ratio = args.balance_ratio
        self.balanced_sampler = args.balanced_sampler
        splits = ['train', 'test', 'val']

        self.creat_dataset_dict(args, splits)

        self.creat_dataloader_map(args, splits)

    def creat_dataset_dict(self, args, splits):
        for task_key in self.task_keys:
            for split in splits:
                dataset = get_dataset(args, task_key, split, self.tokenizer)
                self.encoded_dataset[task_key][split] = dataset

    def creat_dataloader_map(self, args, splits):
        for task_key in self.encoded_dataset:
            for split in splits:
                shuffle = split == 'train'

                batch_size = args.train_batch_size if split == 'train' else args.predict_batch_size
                if split == 'train' and self.few_shot:
                    self.sample_k_shot(self.encoded_dataset[task_key][split])
                # elif split == 'train':
                #     if self.train_limit != -1:
                #         self.trim_subset(
                #             self.encoded_dataset[task_key][split], self.train_limit)
                data = self.encoded_dataset[task_key][split]

                # weighted sampler to handle heavy dataset imbalance
                pos_ratio = np.sum(data.targets) / len(data)
                if self.balance_ratio > 0 and split == 'train' and pos_ratio < self.balance_ratio:
                    # print("fuck")
                    # logger.info(
                    #     f"Using weighted random sampler for Task {task_key}, positive ratio ={pos_ratio}")
                    # # 50/50
                    perfect_balance_weights = [
                        1.0/(1-pos_ratio), 1.0/pos_ratio]
                    class_wieghts = [(1-self.balance_ratio)*perfect_balance_weights[0],
                                    self.balance_ratio*perfect_balance_weights[1]]
                    sample_weights = [class_wieghts[t] for t in data.targets]
                    
                    w_sampler = WeightedRandomSampler(
                        sample_weights, len(data.targets), replacement=True)
                    data_loader = DataLoader(
                        data, batch_size=batch_size, sampler=w_sampler)
                else:
                    data_loader = DataLoader(
                            data, shuffle=shuffle, batch_size=batch_size)
                self.data_loader_maps[task_key][split] = data_loader

    def get_mtl_dataloader(self, split):
        dummy_dataset = list(self.encoded_dataset.values())[0][split]
        shuffle = split == 'train'
        mtl_bal_sampling = split == 'train'
        batch_size = self.args.train_batch_size if split == 'train' else self.args.predict_batch_size
        data_loaders = []
        for task_id, task_key in enumerate(self.data_loader_maps):
            if task_id < len(self.task_keys):
                data_loader = self.data_loader_maps[task_key][split]
                data_loaders.append(data_loader)

        mtl_dataloader = MTLDataloader(dummy_dataset, shuffle=shuffle, batch_size=batch_size,
                                       data_loaders=data_loaders, task_keys=self.task_keys, mtl_bal_sampling=mtl_bal_sampling,
                                       task_num=len(self.task_keys), pad_token_id=self.tokenizer.pad_token_id, sqrt=self.args.sqrt
                                       )
        return mtl_dataloader
    
    def get_dataloader_sequence_iterator(self):
        for task_key in self.data_loader_maps:
            if 'val' in self.data_loader_maps[task_key]:
                data_loaders = [self.data_loader_maps[task_key][split] for split in ['train', 'val', 'test']]
            else:
                data_loaders = [self.data_loader_maps[task_key][split] for split in ['train', 'test', 'test']]
                data_loaders = [self.data_loader_maps[task_key][split] for split in ['train', 'test', 'test']]
            yield task_key, data_loaders



In [12]:
from transformers import AutoTokenizer, RobertaConfig
from mtl_model import MTLRobertaForSequenceClassification
from mtl_main import mtl_train, evaluate
import argparse
import torch
from transformers import AdamW
class Args:
    def __init__(self, label, text_col, dataset, model_name, k_shot, mtl_task_num):
        self.label = label
        self.text_col = text_col
        self.dataset = dataset
        self.model_name = model_name
        self.k_shot = k_shot
        self.mtl_task_num = mtl_task_num


tasks = ["Ann1", "Ann2", "Ann3", "Ann4", "Ann5", "Ann6"]  # , , 

# Create an instance of Args and assign values
args = Args(label='Hate', text_col='text', dataset='brexit',
            mtl_task_num=len(tasks),  model_name='roberta-base', k_shot=32)

model_name = args.model_name


def merge_args_into_config(tasks, config):
    config.tasks = tasks


config = RobertaConfig.from_pretrained(
    'roberta-base', num_labels=2)  # add label2id id2label!

merge_args_into_config(tasks, config)

model = MTLRobertaForSequenceClassification.from_pretrained(
    model_name, config=config)

tokenizer = AutoTokenizer.from_pretrained(model_name)

args.balance_ratio = 0.5
args.train_batch_size = 64
args.predict_batch_size = 16
args.balanced_sampler = True
args.sqrt = False
main_tasks = Tasks(args=args, task_keys=tasks,
                   tokenizer=tokenizer, few_shot=False)

mtl_train_dataloader = main_tasks.get_mtl_dataloader(
    split='train')

args.lr = 1e-5
device = torch.device('cuda:5')
torch.manual_seed(0)
optimizer = AdamW(model.parameters(), lr=args.lr)
model.train()
model.to(device)

model, loss = mtl_train(args, model, mtl_train_dataloader, device, optimizer)

model.eval()
task_iterator = main_tasks.get_dataloader_sequence_iterator()
for task_id, (task_name, (train_loader, dev_loader, test_loader)) in enumerate(task_iterator):
    model.set_classification_head(task_name)
    precision, recall, f1, auc, test_loss = evaluate(model, dev_loader, device)
    print(task_name,  f1, auc, test_loss, precision, recall)

Some weights of MTLRobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classification_heads_dict.Ann2.dense.bias', 'classification_heads_dict.Ann4.out_proj.bias', 'classification_heads_dict.Ann4.dense.weight', 'classification_heads_dict.Ann4.out_proj.weight', 'classification_heads_dict.Ann3.out_proj.weight', 'classifier.dense.weight', 'classification_heads_dict.Ann3.dense.weight', 'classification_heads_dict.Ann5.out_proj.bias', 'classification_heads_dict.Ann6.dense.weight', 'classification_heads_dict.Ann6.out_proj.bias', 'classification_heads_dict.Ann5.dense.bias', 'classification_heads_dict.Ann3.out_proj.bias', 'classification_heads_dict.Ann2.out_proj.weight', 'classification_heads_dict.Ann6.out_proj.weight', 'classification_heads_dict.Ann1.dense.bias', 'classifier.dense.bias', 'classification_heads_dict.Ann5.out_proj.weight', 'classification_heads_dict.Ann1.out_proj.bias', 'classification_heads_dict.Ann1.out_pro

fuck you!
fuck you!
fuck you!
fuck you!
fuck you!
fuck you!


/home/golazizi/2DIS/transformers/src/transformers/optimization.py:423: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
78it [00:09,  7.88it/s]


Ann1 0.19999999999999998 0.8518518518518519 0.5348373055458069 0.1111111111111111 1.0
Ann2 0.25806451612903225 0.85625 0.5116387470202013 0.14814814814814814 1.0
Ann3 0.30769230769230765 0.8144180660104228 0.5124388228763234 0.18518518518518517 0.9090909090909091
Ann4 0.7021276595744681 0.83046875 0.391983072866093 0.6111111111111112 0.825
Ann5 0.6666666666666667 0.8257575757575758 0.42103470184586267 0.5555555555555556 0.8333333333333334
Ann6 0.5647058823529412 0.7776077230986579 0.45255561443892395 0.4444444444444444 0.7741935483870968
